# 00 — Setup and readiness check

Run once, before notebooks 04–09. Locates the repository on Google Drive, verifies the
files each notebook needs, checks that the prediction files are row-aligned with the
test split, and writes the shared path config the other notebooks read.

- Input: a Google Drive copy of this repository
- Output: `MyDrive/r2_config.json`
- Runtime: ~1 min, CPU

## 1. Mount Drive and locate the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, subprocess, sys
from pathlib import Path

MYDRIVE = Path('/content/drive/MyDrive')

# ---- 1. locate the project -------------------------------------------
MARKER = Path('data/cleaned/test_cleaned.json')
candidates = []
for p in MYDRIVE.rglob('test_cleaned.json'):
    root = p.parent.parent.parent          # .../<root>/data/cleaned/test_cleaned.json
    if (root / MARKER).exists():
        candidates.append(root)

candidates = sorted(set(candidates))
if not candidates:
    raise FileNotFoundError(
        'Could not find data/cleaned/test_cleaned.json anywhere in MyDrive.\n'
        'Upload the repository to Drive (keeping its folder structure), or set REPO_DIR manually '
        'in the next cell. The fastest way to get it there is the git clone cell below.')

if len(candidates) > 1:
    print('Multiple project folders found -- using the first. Edit REPO_DIR below if wrong:')
    for c in candidates:
        print('  ', c)

REPO_DIR = str(candidates[0])
RESULTS_DIR = str(Path(REPO_DIR) / 'results_r2')
print('\nREPO_DIR   =', REPO_DIR)
print('RESULTS_DIR=', RESULTS_DIR)


## 2. Optional — clone the repository into Drive

In [ ]:
# OPTIONAL -- only if the search above found nothing.
CLONE_IT = False
if CLONE_IT:
    target = MYDRIVE / 'kazakh-ai-text-detection'
    if target.exists():
        print('already exists:', target)
    else:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/hardkazakh/kazakh-ai-text-detection.git',
                        str(target)], check=True)
        print('cloned to', target)
    REPO_DIR = str(target)
    RESULTS_DIR = str(target / 'results_r2')
    print('REPO_DIR =', REPO_DIR)


## 3. Verify the required files

In [ ]:
# ---- 2. verify the files each notebook needs -------------------------
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas'], check=False)
import pandas as pd

REPO = Path(REPO_DIR)
NEEDED = {
    'data/cleaned/train_cleaned.json': 'notebooks 04, 05, 06, 07, 08, 09',
    'data/cleaned/dev_cleaned.json':   'notebooks 04, 05, 06, 07, 08, 09',
    'data/cleaned/test_cleaned.json':  'notebooks 04, 05, 06, 07, 08, 09',
    'results/predictions/preds_mDeBERTa-v3_base.csv': 'notebooks 04, 05',
    'results/predictions/preds_XLM-R_base.csv':       'notebooks 04, 05',
    'results/predictions/preds_mBERT_cased.csv':      'notebooks 04, 05',
    'results/predictions/preds_DistilmBERT.csv':      'notebooks 04, 05',
    'results/predictions/preds_TF-IDF_+_LogReg.csv':  'notebooks 04, 05',
}
rows = []
for rel, used_by in NEEDED.items():
    ok = (REPO / rel).exists()
    rows.append({'file': rel, 'present': 'YES' if ok else 'MISSING', 'used_by': used_by})
files_df = pd.DataFrame(rows)
print(files_df.to_string(index=False))
missing = files_df[files_df.present == 'MISSING']
if len(missing):
    print('\n*** Missing files -- the listed notebooks will fail. ***')


## 4. Verify prediction/test row alignment

In [ ]:
# ---- 3. THE critical precondition ------------------------------------
# Everything in notebooks 04 and 05 depends on prediction rows lining up
# with test rows. If this fails, every re-sliced metric would be garbage.
import numpy as np

test = pd.DataFrame(json.load(open(REPO / 'data/cleaned/test_cleaned.json', encoding='utf-8')))
y_test = test['label'].astype(int).to_numpy()

aligned = True
for f in sorted((REPO / 'results/predictions').glob('preds_*.csv')):
    p = pd.read_csv(f)
    ok = len(p) == len(test) and (p['y_true'].to_numpy() == y_test).all()
    aligned &= ok
    print(f'  {"OK  " if ok else "FAIL"}  {f.name}')

print()
if aligned:
    print('Prediction/test row alignment VERIFIED.')
    print('This is what lets notebooks 04 and 05 re-score your existing models on')
    print('length-matched and near-duplicate-free subsets without retraining anything.')
else:
    print('*** ALIGNMENT FAILED. Do not trust notebook 04/05 output until this is fixed. ***')

print('\nTest set:', len(test), 'documents |',
      dict(pd.Series(y_test).map({0: 'Human', 1: 'AI-Gen', 2: 'AI-Obf'}).value_counts()))


## 5. Report GPU and runtime estimates

In [ ]:
# ---- 4. what hardware did Colab give you? ----------------------------
gpu_name, gpu_mem = None, None
try:
    out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader'], capture_output=True, text=True)
    if out.returncode == 0 and out.stdout.strip():
        gpu_name, gpu_mem = [x.strip() for x in out.stdout.strip().split('\n')[0].split(',')]
except FileNotFoundError:
    pass

print('GPU:', gpu_name or 'NONE (Runtime > Change runtime type > GPU)')
if gpu_mem:
    print('VRAM:', gpu_mem)

# runtime estimates, scaled from the per-run cost of one base-sized encoder
PROFILE = {
    'T4':   {'min_per_seed': 28, 'gen_per_100': 45, 'note': 'free tier; no bf16, so generation is slow'},
    'L4':   {'min_per_seed': 14, 'gen_per_100': 18, 'note': 'Colab Pro; comfortable for everything'},
    'A100': {'min_per_seed': 7,  'gen_per_100': 8,  'note': 'Pro+; fastest, but burns compute units'},
    'V100': {'min_per_seed': 18, 'gen_per_100': 30, 'note': 'older Pro GPU'},
}
key = next((k for k in PROFILE if gpu_name and k in gpu_name), None)
if key:
    pr = PROFILE[key]
    print(f'\nEstimates for {key} ({pr["note"]}):')
    print(f'  notebook 09, 5 seeds x 4 models  (20 runs): ~{20*pr["min_per_seed"]/60:.1f} h')
    print(f'  notebook 09, 10/10/5/5 seeds     (30 runs): ~{30*pr["min_per_seed"]/60:.1f} h')
    print(f'  notebook 08, 345 generations             : ~{345/100*pr["gen_per_100"]/60:.1f} h')
    print(f'  notebook 07 Route B, 600 paraphrases     : ~{600/100*pr["gen_per_100"]/60:.1f} h')
    print('\nThese are rough. Notebooks 08 and 09 are resumable, so a disconnect is not a restart.')
else:
    print('\nUnrecognised or absent GPU -- notebooks 04, 05, 06 still run fine on CPU.')


## 6. Check model-host reachability

In [ ]:
# ---- 5. can Colab reach the model hosts? -----------------------------
import urllib.request
def reachable(url, timeout=12):
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return True
    except Exception as e:
        return f'{type(e).__name__}'

checks = {
    'huggingface.co (tokenizers, generators)': 'https://huggingface.co',
    'kaggle.com (KerasNLP presets)':           'https://www.kaggle.com',
    'storage.googleapis.com (weights CDN)':    'https://storage.googleapis.com',
}
for label, url in checks.items():
    r = reachable(url)
    print(f'  {"OK  " if r is True else "FAIL"}  {label}  {"" if r is True else r}')

print('\nAll three are normally reachable from Colab. If any fails, retry -- it is usually transient.')

# Drive space -- notebook 09 saves ~1.1 GB per checkpoint
st = os.statvfs('/content/drive/MyDrive')
free_gb = st.f_bavail * st.f_frsize / 1e9
print(f'\nDrive free space: ~{free_gb:.1f} GB')
if free_gb < 6:
    print('WARNING: notebook 09 saves ~3.3 GB of weights (3 checkpoints) for notebook 08 to reuse.')
    print('Either free up space, or set SAVE_WEIGHTS_FOR = {} and let notebook 08 retrain instead.')


## 7. Write the shared path config

In [ ]:
# ---- 6. write the shared config every other notebook reads -----------
cfg = {
    'REPO_DIR': REPO_DIR,
    'RESULTS_DIR': RESULTS_DIR,
    'RESULTS_SEEDS_DIR': str(Path(REPO_DIR) / 'results_r2_seeds'),
    'RESULTS_PAIRED_DIR': str(Path(REPO_DIR) / 'results_r2_paired'),
    'RESULTS_CROSSGEN_DIR': str(Path(REPO_DIR) / 'results_r2_crossgen'),
    'WEIGHTS_DIR': str(Path(REPO_DIR) / 'results_r2_seeds' / 'weights'),
    'gpu': gpu_name,
}
for k, v in cfg.items():
    if k.endswith('_DIR'):
        Path(v).mkdir(parents=True, exist_ok=True)

cfg_path = MYDRIVE / 'r2_config.json'
json.dump(cfg, open(cfg_path, 'w'), indent=2)
print('wrote', cfg_path)
print(json.dumps(cfg, indent=2))
print('\nEvery other notebook auto-loads this file. You should not need to edit a path again.')
